# Log e Registro do modelo via MLFlow Model Serving

## Log do modelo como parte do experimento `heart-disease-prediction`

In [0]:
import warnings, mlflow, joblib
import pandas as pd
from mlflow.models.signature import infer_signature

warnings.filterwarnings('ignore')

# Carregar o modelo via joblib
model = joblib.load('models/best_random_forest.joblib')

# Criar experimento
mlflow.set_experiment('/heart-disease-prediction')

# Carregar dataset de exemplo
path = "../../data/heart_disease_uci_preprocessed.csv"
df = pd.read_csv(path)

# Selecionar uma amostra de exemplo
X_train = df.drop(columns=['target'])[:5]
y_train = df['target'][:5]

# Adicionar assinatura do modelo
model_signature = infer_signature(X_train, y_train)

# Log do modelo no MLFlow
mlflow_model = mlflow.sklearn.log_model(model, 'model', signature=model_signature)
mlflow.end_run()

## Registro do modelo

In [0]:
from mlflow import MlflowClient

# Set Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Registrar modelo
model_registered = mlflow.register_model(model_uri = mlflow_model.model_uri,
                                         name = 'heart-disease-model')


# Criar alias Staging/Production e setar ao modelo
MlflowClient().set_registered_model_alias(model_registered.name, 
                                  "Production", 
                                  model_registered.version)


## Configurar o modelo atual como Staging

In [0]:
from mlflow import MlflowClient
client = MlflowClient()

# Criar alias Staging e setar ao modelo
client.set_registered_model_alias(model_registered.name, 
                                  "Staging", 
                                  model_registered.version)

### Pré-processamento para exemplo de Serving
Abaixo adaptamos a rotina de pré-processamento usada na aplicação Flask (flask-app/preprocessing.py) e a utilizamos também no exemplo de requisição com MLFlow Serving:
- Carregamos uma linha *raw* do CSV original, aplicamos o pré-processamento e chamamos o modelo logado no MLflow.
- Isso demonstra o mesmo fluxo que a aplicação Flask usa antes de chamar o modelo.

In [0]:
# Código de pré-processamento (adaptado de flask-app/preprocessing.py)
import pandas as pd
import numpy as np
from typing import Optional, List

class HeartDiseasePreprocessor:
    def __init__(self) -> None:
        pass

    def apply_feature_engineering(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        eps = 1

        if 'age' in out.columns:
            if 'age_squared' not in out.columns:
                out['age_squared'] = out['age'] ** 2
            if 'age_decade' not in out.columns:
                out['age_decade'] = (out['age'] // 10).astype(int)

        if 'chol' in out.columns and 'age' in out.columns and 'cholesterol_to_age' not in out.columns:
            out['cholesterol_to_age'] = out['chol'] / (out['age'] + eps)

        if 'thalch' in out.columns and 'age' in out.columns and 'max_hr_pct' not in out.columns:
            predicted_max_hr = (220 - out['age']).clip(lower=1)
            out['max_hr_pct'] = out['thalch'] / (predicted_max_hr + eps)

        if 'trestbps' in out.columns and 'chol' in out.columns and 'bp_chol_ratio' not in out.columns:
            out['bp_chol_ratio'] = out['trestbps'] / (out['chol'] + 1)

        if 'fbs' in out.columns and 'fbs_flag' not in out.columns:
            out['fbs_flag'] = out['fbs'].astype(int)

        if 'exang' in out.columns and 'exang_flag' not in out.columns:
            out['exang_flag'] = out['exang'].astype(int)

        if 'thalch' in out.columns and 'trestbps' in out.columns and 'stress_index' not in out.columns:
            out['stress_index'] = out['thalch'] / (out['trestbps'] + eps)

        if 'age' in out.columns and 'oldpeak' in out.columns and 'risk_interaction' not in out.columns:
            out['risk_interaction'] = out['age'] * out['oldpeak']

        if 'oldpeak' in out.columns and 'high_st_depression_flag' not in out.columns:
            out['high_st_depression_flag'] = (out['oldpeak'] > 1.0).astype(int)

        for c in out.columns:
            if pd.api.types.is_object_dtype(out[c]):
                try:
                    out[c] = pd.to_numeric(out[c])
                except Exception:
                    pass

        if 'target' in out.columns:
            out = out.drop(columns=['target'])

        return out

    def apply_raw_categorical_encoding(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        if 'dataset' in out.columns:
            out = out.drop(columns=['dataset'])

        def map_col(col: str, mapping: dict):
            if col in out.columns:
                series = out[col]
                if series.dtype == bool:
                    numeric = series.astype(int)
                else:
                    try:
                        numeric = pd.to_numeric(series)
                    except Exception:
                        numeric = series

                if pd.api.types.is_numeric_dtype(numeric):
                    vals = numeric.astype(int).values
                    out[col] = [mapping.get(int(v), mapping.get(v, series.iloc[i] if i < len(series) else v))
                                for i, v in enumerate(vals)]
                else:
                    inv = {v: v for v in mapping.values()}
                    out[col] = [inv.get(str(x), str(x)) for x in series]

        sex_map = {0: 'Female', 1: 'Male'}
        cp_map = {0: 'typical angina', 1: 'atypical angina', 2: 'non-anginal', 3: 'asymptomatic'}
        rest_map = {0: 'normal', 1: 'st-t abnormality', 2: 'left ventricular hypertrophy'}
        slope_map = {0: 'upsloping', 1: 'flat', 2: 'downsloping'}
        thal_map = {0: 'normal', 1: 'fixed defect', 2: 'reversable defect'}

        map_col('sex', sex_map)
        map_col('cp', cp_map)
        map_col('restecg', rest_map)
        map_col('slope', slope_map)
        map_col('thal', thal_map)

        categorical_cols = [c for c in ['sex', 'cp', 'restecg', 'slope', 'thal'] if c in out.columns]
        if categorical_cols:
            out = pd.get_dummies(out, columns=categorical_cols, drop_first=True)

        return out

    def align_features(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.expected_feature_order is None:
            return df
        aligned = df.copy()
        for col in self.expected_feature_order:
            if col not in aligned.columns:
                aligned[col] = 0
        aligned = aligned[self.expected_feature_order]
        for c in aligned.columns:
            if aligned[c].dtype == bool:
                aligned[c] = aligned[c].astype(int)
        return aligned

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = self.apply_raw_categorical_encoding(df)
        out = self.apply_feature_engineering(out)
        out = self.align_features(out)
        return out

In [0]:
# Instanciar o pré-processador com o modelo carregado anteriormente
preprocessor = HeartDiseasePreprocessor()

# Carregar uma linha raw do CSV original e simular uma requisição JSON
raw_df = pd.read_csv('../../data/heart_disease_uci_preprocessed.csv')
raw_row = raw_df.drop(columns=['target'], errors='ignore').iloc[[0]]

In [0]:
import requests
import json

TOKEN = ""

# Serializar a linha raw para JSON
payload = {
    "data": raw_row.to_dict(orient="records")
}

print(json.dumps(payload))

# URL do endpoint de serving do modelo (ajuste conforme necessário)
endpoint_url = "https://<>.cloud.databricks.com/serving-endpoints/heart-disease-predict/invocations"

# Cabeçalhos típicos para Databricks Model Serving
headers = {
    f"Authorization": "Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Enviar requisição POST
response = requests.post(endpoint_url, headers=headers, data=json.dumps(payload))

# Exibir resposta
print("Status code:", response.status_code)
print("Resposta do modelo:", response.json())